# D11 Calculation Pathway Effects

Analyze how `D11` changes across calculation pathways while holding the observed ECTP slab fixed. 

In [16]:
import math
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from notebook_utils import create_ectp_slabs, hess_rcparams, load_pits, nominal
from paper_figure_utils import (
    build_d11_paired_pathway_effects_figure,
    build_d11_spread_attribute_correlations_figure,
    format_method_path,
    notebook_latex,
    save_paper_figure,
)
from snowpyt_mechparams.execution import ExecutionEngine
from snowpyt_mechparams.execution.config import ExecutionConfig
from snowpyt_mechparams.graph import default_graph
from snowpyt_mechparams.pathway import find_parameterizations

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(items, **_kwargs):
        return items

hess_rcparams()

TOP_N = 8

## Helper Functions

Keep notebook helpers small and explicit so the paired calculation is easy to audit.

In [17]:
def finite_array(values) -> np.ndarray:
    """Return finite float values from an iterable of plain or uncertain values."""
    arr = np.asarray([nominal(value) for value in values], dtype=float)
    return arr[np.isfinite(arr)]


def finite_positive(value: float) -> bool:
    """Return True for finite, positive values."""
    return math.isfinite(value) and value > 0


def coefficient_of_variation(values: np.ndarray) -> float:
    """Return std / mean, or NaN when the mean is unavailable or zero."""
    if values.size == 0:
        return math.nan
    mean_value = float(np.mean(values))
    if mean_value == 0 or not math.isfinite(mean_value):
        return math.nan
    return float(np.std(values, ddof=0) / abs(mean_value))


def pathway_key(methods: dict[str, str]) -> str:
    """Return the density -> E -> nu pathway key used in paper tables."""
    return ' -> '.join(
        [
            methods.get('density', 'data_flow'),
            methods.get('elastic_modulus', 'unknown'),
            methods.get('poissons_ratio', 'unknown'),
        ]
    )


def extract_d11(pathway_result) -> tuple[float, float]:
    """Return nominal D11 and standard deviation from a pathway result."""
    for trace in pathway_result.computation_trace:
        if trace.parameter == 'D11' and trace.success and trace.output is not None:
            output = trace.output
            return nominal(output), float(getattr(output, 'std_dev', 0.0))
    return math.nan, math.nan


def slab_attribute_record(slab_index: int, slab) -> dict[str, float | int | str | None]:
    """Summarize slab geometry and layer attributes used in spread analysis."""
    thicknesses = finite_array(layer.thickness for layer in slab.layers)
    hand_hardness = finite_array(layer.hand_hardness_index for layer in slab.layers)

    weighted_pairs = []
    for layer in slab.layers:
        thickness = nominal(layer.thickness)
        hhi = nominal(layer.hand_hardness_index)
        if math.isfinite(thickness) and thickness > 0 and math.isfinite(hhi):
            weighted_pairs.append((thickness, hhi))

    if weighted_pairs:
        weights = np.asarray([pair[0] for pair in weighted_pairs], dtype=float)
        hhi_values = np.asarray([pair[1] for pair in weighted_pairs], dtype=float)
        weighted_hhi = float(np.average(hhi_values, weights=weights))
    else:
        weighted_hhi = math.nan

    return {
        'slab_index': slab_index,
        'slab_id': slab.slab_id,
        'pit_id': slab.pit_id,
        'n_layers': len(slab.layers),
        'total_thickness_cm': nominal(slab.total_thickness),
        'slope_angle_deg': nominal(slab.angle),
        'layer_thickness_mean_cm': float(np.mean(thicknesses)) if thicknesses.size else math.nan,
        'layer_thickness_std_cm': float(np.std(thicknesses, ddof=0)) if thicknesses.size else math.nan,
        'layer_thickness_cv': coefficient_of_variation(thicknesses),
        'layer_thickness_max_cm': float(np.max(thicknesses)) if thicknesses.size else math.nan,
        'hand_hardness_index_mean': float(np.mean(hand_hardness)) if hand_hardness.size else math.nan,
        'hand_hardness_index_std': float(np.std(hand_hardness, ddof=0)) if hand_hardness.size else math.nan,
        'hand_hardness_index_range': float(np.ptp(hand_hardness)) if hand_hardness.size else math.nan,
        'hand_hardness_index_cv': coefficient_of_variation(hand_hardness),
        'hand_hardness_index_weighted_mean': weighted_hhi,
    }


def scientific_latex(value: float) -> str:
    """Return a compact LaTeX scientific-notation value."""
    if not math.isfinite(value):
        return ''
    coefficient, exponent = f'{value:.3e}'.split('e')
    return rf'${coefficient} \times 10^{{{int(exponent)}}}$'


def formatted_pathway(pathway: str, *, short: bool = False) -> str:
    """Return a paper-friendly label for a pathway key."""
    return format_method_path(*pathway.split(' -> '), short=short)

## Load Data and Enumerate D11 Pathways

Use the same ECTP slab definition as the slab-weight input analysis: Slab is the set of layers above the ECTP layer of propagation

In [18]:
pits = load_pits()
ectp_slabs = create_ectp_slabs(pits)
total_slabs = len(ectp_slabs)

pathways = find_parameterizations(default_graph, default_graph.get_node('D11'))
engine = ExecutionEngine()
config = ExecutionConfig(include_method_uncertainty=False) # input uncertainty only

print(f'Loaded {len(pits):,} pits and {total_slabs:,} ECTP slabs')
print(f'D11 pathways: {len(pathways)}')

Loaded 50,278 pits and 14,776 ECTP slabs
D11 pathways: 32


## Execute D11 Pathways

Store one row per slab-pathway attempt. Failed attempts remain in `d11_results` with `NaN` values so coverage can be computed directly from the same table.

In [19]:
records = []

for slab_index, slab in enumerate(ectp_slabs):
    slab_attrs = slab_attribute_record(slab_index, slab)
    results = engine.execute_all(slab, 'D11', config=config, pathways=pathways)

    for pathway_result in results.pathways.values():
        methods = pathway_result.methods_used
        d11_nominal, d11_std = extract_d11(pathway_result)
        d11_success = finite_positive(d11_nominal)

        records.append(
            {
                **slab_attrs,
                'pathway': pathway_key(methods),
                'pathway_description': pathway_result.pathway_description,
                'density_method': methods.get('density', 'data_flow'),
                'emod_method': methods.get('elastic_modulus', 'unknown'),
                'nu_method': methods.get('poissons_ratio', 'unknown'),
                'pathway_result_success': pathway_result.success,
                'success': d11_success,
                'D11_nominal': d11_nominal if d11_success else math.nan,
                'D11_std': d11_std if d11_success else math.nan,
                'D11_rel_uncertainty': d11_std / abs(d11_nominal) if d11_success else math.nan,
            }
        )

d11_results = pd.DataFrame(records)
d11_success = d11_results[d11_results['success']].copy()

print(f'Pathway attempts: {len(d11_results):,}')
print(f'Successful D11 calculations: {len(d11_success):,}')
print(f'Slabs with at least one successful D11 value: {d11_success["slab_index"].nunique():,}')
d11_results.head()

Pathway attempts: 472,832
Successful D11 calculations: 8,858
Slabs with at least one successful D11 value: 823


,slab_index,slab_id,pit_id,n_layers,total_thickness_cm,slope_angle_deg,layer_thickness_mean_cm,layer_thickness_std_cm,layer_thickness_cv,layer_thickness_max_cm,...,pathway,pathway_description,density_method,emod_method,nu_method,pathway_result_success,success,D11_nominal,D11_std,D11_rel_uncertainty
0,0,1000_slab_0,1000,5,28.0,30.0,5.6,5.571355,0.994885,15.0,...,data_flow -> bergfeld -> kochle,density=data_flow | elastic_modulus=bergfeld |...,data_flow,bergfeld,kochle,False,False,NaN,NaN,NaN
1,0,1000_slab_0,1000,5,28.0,30.0,5.6,5.571355,0.994885,15.0,...,data_flow -> bergfeld -> srivastava,density=data_flow | elastic_modulus=bergfeld |...,data_flow,bergfeld,srivastava,False,False,NaN,NaN,NaN
2,0,1000_slab_0,1000,5,28.0,30.0,5.6,5.571355,0.994885,15.0,...,geldsetzer -> bergfeld -> srivastava,density=geldsetzer | elastic_modulus=bergfeld ...,geldsetzer,bergfeld,srivastava,False,False,NaN,NaN,NaN
3,0,1000_slab_0,1000,5,28.0,30.0,5.6,5.571355,0.994885,15.0,...,kim_jamieson_table2 -> bergfeld -> srivastava,density=kim_jamieson_table2 | elastic_modulus=...,kim_jamieson_table2,bergfeld,srivastava,False,False,NaN,NaN,NaN
4,0,1000_slab_0,1000,5,28.0,30.0,5.6,5.571355,0.994885,15.0,...,kim_jamieson_table5 -> bergfeld -> srivastava,density=kim_jamieson_table5 | elastic_modulus=...,kim_jamieson_table5,bergfeld,srivastava,False,False,NaN,NaN,NaN


## Top 8 Coverage Pathways

Rank pathways by successful ECTP slab coverage. 

In [20]:
pathway_coverage = (
    d11_results.groupby(['pathway', 'density_method', 'emod_method', 'nu_method'])
    .agg(
        successful_slabs=('success', 'sum'),
        attempted_slabs=('success', 'count'),
    )
    .reset_index()
    .sort_values(['successful_slabs', 'pathway'], ascending=[False, True])
)
pathway_coverage['coverage_percent'] = 100.0 * pathway_coverage['successful_slabs'] / total_slabs

top_pathways = pathway_coverage.head(TOP_N)['pathway'].tolist()
coverage_table = pd.DataFrame(
    {
        'Pathway': [formatted_pathway(pathway) for pathway in top_pathways],
        'Successful slabs': pathway_coverage.head(TOP_N)['successful_slabs'].map(lambda value: f'{int(value):,}'),
        'Coverage (%)': pathway_coverage.head(TOP_N)['coverage_percent'].map(lambda value: f'{value:.1f}'),
    }
)

coverage_table

# Update index to be ranking
coverage_table.index = coverage_table.index.get_indexer(coverage_table.index)
coverage_table.index.name = 'Rank'
coverage_table


,Pathway,Successful slabs,Coverage (%)
Rank,,,
0,Kim and Jamieson Table 2 -> Schottner et al. (...,743,5.0
1,Kim and Jamieson Table 2 -> Wautier et al. (20...,743,5.0
2,Geldsetzer and Jamieson (2000) -> Schottner et...,740,5.0
3,Geldsetzer and Jamieson (2000) -> Wautier et a...,740,5.0
4,Geldsetzer and Jamieson (2000) -> Kochle and S...,733,5.0
5,Kim and Jamieson Table 2 -> Kochle and Schneeb...,634,4.3
6,Kim and Jamieson Table 2 -> Bergfeld et al. (2...,498,3.4
7,Geldsetzer and Jamieson (2000) -> Bergfeld et ...,475,3.2


## Paired Top-8 Slab Subset

Keep only slabs where all top-8 pathways produced finite, positive D11 values. This creates a paired matrix where each row is the same slab observed through different calculation pathways.

In [ ]:
top8_success = d11_success[d11_success['pathway'].isin(top_pathways)].copy()
common_counts = top8_success.groupby('slab_index')['pathway'].nunique()
common_slab_indices = common_counts[common_counts == TOP_N].index

d11_common = top8_success[top8_success['slab_index'].isin(common_slab_indices)].copy()
d11_common['pathway'] = pd.Categorical(d11_common['pathway'], categories=top_pathways, ordered=True)
d11_common = d11_common.sort_values(['pathway', 'slab_index'])

d11_wide = d11_common.pivot(
    index=['slab_index', 'slab_id', 'pit_id'],
    columns='pathway',
    values='D11_nominal',
).reindex(columns=top_pathways)

within_slab_median = d11_wide.median(axis=1)
ratio_wide = d11_wide.divide(within_slab_median, axis=0)

paired_effects = (
    ratio_wide.stack()
    .rename('pathway_ratio')
    .reset_index()
    .merge(
        d11_common[
            ['slab_index', 'slab_id', 'pit_id', 'pathway', 'D11_nominal', 'D11_rel_uncertainty']
        ],
        on=['slab_index', 'slab_id', 'pit_id', 'pathway'],
        how='left',
    )
)
paired_effects['within_slab_median_D11'] = paired_effects.set_index(
    ['slab_index', 'slab_id', 'pit_id']
).index.map(within_slab_median)
paired_effects['pathway_percent_difference'] = 100.0 * (paired_effects['pathway_ratio'] - 1.0)
paired_effects['log10_pathway_ratio'] = np.log10(paired_effects['pathway_ratio'])
paired_effects['pathway'] = pd.Categorical(paired_effects['pathway'], categories=top_pathways, ordered=True)
paired_effects = paired_effects.sort_values(['pathway', 'slab_index'])

print(f'Top-8 pathways: {TOP_N}')
print(f'Slabs successful for all top-8 pathways: {len(common_slab_indices):,}')
paired_effects[['slab_id', 'pathway', 'pathway_ratio', 'pathway_percent_difference']].head()

# Define pathway_ratio and pathway_percent_difference


Top-8 pathways: 8
Slabs successful for all top-8 pathways: 412


,slab_id,pathway,pathway_ratio,pathway_percent_difference
0,1036_slab_0,kim_jamieson_table2 -> schottner -> kochle,0.161971,-83.802903
8,1057_slab_0,kim_jamieson_table2 -> schottner -> kochle,0.193922,-80.607758
16,1057_slab_1,kim_jamieson_table2 -> schottner -> kochle,0.193922,-80.607758
24,1057_slab_2,kim_jamieson_table2 -> schottner -> kochle,0.193922,-80.607758
32,1057_slab_3,kim_jamieson_table2 -> schottner -> kochle,0.193922,-80.607758


## Paired Pathway Effects

Express each pathway's D11 as a ratio to the median top-8 pathway result for the same slab. A ratio of 1 means that pathway equals the within-slab median.

In [22]:
def percentile(values, q: float) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.percentile(arr, q)) if arr.size else math.nan


paired_effect_summary = (
    paired_effects.groupby('pathway', observed=True)
    .agg(
        common_slabs=('slab_index', 'nunique'),
        median_ratio=('pathway_ratio', 'median'),
        ratio_p05=('pathway_ratio', lambda values: percentile(values, 5)),
        ratio_p95=('pathway_ratio', lambda values: percentile(values, 95)),
        median_percent_difference=('pathway_percent_difference', 'median'),
        median_relative_uncertainty=('D11_rel_uncertainty', 'median'),
    )
    .reindex(top_pathways)
    .reset_index()
)

paired_effect_table = pd.DataFrame(
    {
        'Pathway': [formatted_pathway(str(pathway)) for pathway in paired_effect_summary['pathway']],
        'Common slabs': paired_effect_summary['common_slabs'].map(lambda value: f'{int(value):,}'),
        'Median ratio': paired_effect_summary['median_ratio'].map(lambda value: f'{value:.2f}'),
        '5th-95th ratio': paired_effect_summary.apply(
            lambda row: f'{row["ratio_p05"]:.2f}-{row["ratio_p95"]:.2f}', axis=1
        ),
        'Median difference from slab median (%)': paired_effect_summary['median_percent_difference'].map(lambda value: f'{value:.1f}'),
        'Median relative uncertainty (%)': paired_effect_summary['median_relative_uncertainty'].map(lambda value: f'{100.0 * value:.1f}'),
    }
)

paired_effect_paths = save_paper_figure(
    build_d11_paired_pathway_effects_figure(paired_effects, top_pathways, top_n=TOP_N),
    'd11_paired_pathway_effects',
    close=True,
)

print('Paired pathway-effect exports:', paired_effect_paths)
paired_effect_table

Paired pathway-effect exports: {'repo_pdf': PosixPath('/Users/marykate/Desktop/Snow/SnowPyt-MechParams/paper/figures/d11_paired_pathway_effects.pdf'), 'repo_png': PosixPath('/Users/marykate/Desktop/Snow/SnowPyt-MechParams/paper/figures/d11_paired_pathway_effects.png'), 'external_pdf': PosixPath('/Users/marykate/Desktop/Snow/mech_params_paper/figures/d11_paired_pathway_effects.pdf'), 'external_png': PosixPath('/Users/marykate/Desktop/Snow/mech_params_paper/figures/d11_paired_pathway_effects.png'), 'pdf': PosixPath('/Users/marykate/Desktop/Snow/SnowPyt-MechParams/paper/figures/d11_paired_pathway_effects.pdf'), 'png': PosixPath('/Users/marykate/Desktop/Snow/SnowPyt-MechParams/paper/figures/d11_paired_pathway_effects.png'), 'external_dir': PosixPath('/Users/marykate/Desktop/Snow/mech_params_paper/figures')}


,Pathway,Common slabs,Median ratio,5th-95th ratio,Median difference from slab median (%),Median relative uncertainty (%)
0,Kim and Jamieson Table 2 -> Schottner et al. (...,412,0.21,0.19-0.36,-79.2,84.6
1,Kim and Jamieson Table 2 -> Wautier et al. (20...,412,6.67,5.29-37.31,567.1,44.9
2,Geldsetzer and Jamieson (2000) -> Schottner et...,412,0.22,0.20-0.47,-78.3,89.5
3,Geldsetzer and Jamieson (2000) -> Wautier et a...,412,6.82,5.77-43.01,581.7,42.5
4,Geldsetzer and Jamieson (2000) -> Kochle and S...,412,1.61,1.27-1.69,61.5,70.7
5,Kim and Jamieson Table 2 -> Kochle and Schneeb...,412,1.58,0.92-1.79,57.6,60.7
6,Kim and Jamieson Table 2 -> Bergfeld et al. (2...,412,0.43,0.39-0.83,-57.3,81.0
7,Geldsetzer and Jamieson (2000) -> Bergfeld et ...,412,0.46,0.42-1.08,-54.4,85.7


## Paired Pathway Spread

Summarize how far pathways diverge within each slab. The primary spread metric is `log10(max/min)`, which is unitless and symmetric on a ratio scale; absolute D11 range is retained only as supporting context.

In [23]:
min_pathways = d11_wide.idxmin(axis=1).astype(str)
max_pathways = d11_wide.idxmax(axis=1).astype(str)
slab_attr = d11_common.groupby(['slab_index', 'slab_id', 'pit_id']).agg(
    n_layers=('n_layers', 'first'),
    total_thickness_cm=('total_thickness_cm', 'first'),
    slope_angle_deg=('slope_angle_deg', 'first'),
    layer_thickness_mean_cm=('layer_thickness_mean_cm', 'first'),
    layer_thickness_std_cm=('layer_thickness_std_cm', 'first'),
    layer_thickness_cv=('layer_thickness_cv', 'first'),
    layer_thickness_max_cm=('layer_thickness_max_cm', 'first'),
    hand_hardness_index_mean=('hand_hardness_index_mean', 'first'),
    hand_hardness_index_std=('hand_hardness_index_std', 'first'),
    hand_hardness_index_range=('hand_hardness_index_range', 'first'),
    hand_hardness_index_cv=('hand_hardness_index_cv', 'first'),
    hand_hardness_index_weighted_mean=('hand_hardness_index_weighted_mean', 'first'),
).reset_index()

paired_spread = pd.DataFrame(
    {
        'slab_index': [idx[0] for idx in d11_wide.index],
        'slab_id': [idx[1] for idx in d11_wide.index],
        'pit_id': [idx[2] for idx in d11_wide.index],
        'within_slab_median_D11': within_slab_median.to_numpy(),
        'min_D11': d11_wide.min(axis=1).to_numpy(),
        'max_D11': d11_wide.max(axis=1).to_numpy(),
        'min_pathway': min_pathways.to_numpy(),
        'max_pathway': max_pathways.to_numpy(),
    }
).merge(slab_attr, on=['slab_index', 'slab_id', 'pit_id'], how='left')

paired_spread = paired_spread.assign(
    pathway_spread_ratio=lambda frame: frame['max_D11'] / frame['min_D11'],
    log10_pathway_spread=lambda frame: np.log10(frame['max_D11'] / frame['min_D11']),
    range_D11=lambda frame: frame['max_D11'] - frame['min_D11'],
    relative_range=lambda frame: frame['range_D11'] / frame['within_slab_median_D11'],
    log10_median_D11=lambda frame: np.log10(frame['within_slab_median_D11']),
)

paired_spread[['pathway_spread_ratio', 'log10_pathway_spread', 'range_D11']].describe(
    percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
)

,pathway_spread_ratio,log10_pathway_spread,range_D11
count,412.000000,412.000000,4.120000e+02
mean,49.604937,1.649974,1.223440e+09
std,25.670540,0.188683,3.351700e+09
min,25.507161,1.406662,3.805427e+04
5%,27.500173,1.439335,2.511748e+06
25%,31.541034,1.498876,4.261879e+07
50%,36.639798,1.563953,2.232292e+08
75%,62.496604,1.795856,1.027465e+09
95%,120.430113,2.080735,5.304322e+09
max,120.430113,2.080735,3.222913e+10


## Filter Obvious Outliers

Remove extreme geometry and extreme median D11 values before ranking paired spread: keep the central 95% of total slab thickness and `log10(within-slab median D11)`.

In [24]:
thickness_low, thickness_high = paired_spread['total_thickness_cm'].quantile([0.025, 0.975])
log_median_low, log_median_high = paired_spread['log10_median_D11'].quantile([0.025, 0.975])

outlier_filter = (
    paired_spread['total_thickness_cm'].between(thickness_low, thickness_high)
    & paired_spread['log10_median_D11'].between(log_median_low, log_median_high)
)
paired_spread_filtered = paired_spread[outlier_filter].copy()

print(f'Paired slabs before filtering: {len(paired_spread):,}')
print(f'Paired slabs after geometry + D11 filter: {len(paired_spread_filtered):,}')
print(f'Removed as outliers: {len(paired_spread) - len(paired_spread_filtered):,}')
print(f'Thickness central 95%: {thickness_low:.1f} to {thickness_high:.1f} cm')
print(f'log10(within-slab median D11) central 95%: {log_median_low:.2f} to {log_median_high:.2f}')

Paired slabs before filtering: 412
Paired slabs after geometry + D11 filter: 385
Removed as outliers: 27
Thickness central 95%: 4.0 to 63.0 cm
log10(within-slab median D11) central 95%: 4.73 to 9.12


## Smallest and Widest Paired Spread Slabs

Rank filtered slabs by `log10(max/min)`. Absolute D11 range is included to preserve physical scale, but the ratio-based spread is the paired comparison metric.

In [25]:
def prepare_spread_rank_table(frame: pd.DataFrame, *, top_n: int = 8) -> pd.DataFrame:
    """Format a paired spread ranking table for manuscript copy/paste."""
    return pd.DataFrame(
        {
            'Slab ID': frame['slab_id'].astype(str),
            'Pit ID': frame['pit_id'].astype(str),
            'Layers': frame['n_layers'].astype(int).astype(str),
            'Thickness (cm)': frame['total_thickness_cm'].map(lambda value: f'{value:.1f}'),
            'Min pathway': frame['min_pathway'].map(lambda value: formatted_pathway(value, short=True)),
            'Max pathway': frame['max_pathway'].map(lambda value: formatted_pathway(value, short=True)),
            'Max/min ratio': frame['pathway_spread_ratio'].map(lambda value: f'{value:.1f}'),
            'log10(max/min)': frame['log10_pathway_spread'].map(lambda value: f'{value:.2f}'),
            'D11 range (N·mm)': frame['range_D11'].map(scientific_latex),
        }
    ).head(top_n)


smallest_spread = paired_spread_filtered.sort_values('log10_pathway_spread', ascending=True).head(TOP_N)
widest_spread = paired_spread_filtered.sort_values('log10_pathway_spread', ascending=False).head(TOP_N)
smallest_spread_table = prepare_spread_rank_table(smallest_spread, top_n=TOP_N)
widest_spread_table = prepare_spread_rank_table(widest_spread, top_n=TOP_N)

print('Smallest filtered paired spread slabs:')
display(smallest_spread_table)
print('Widest filtered paired spread slabs:')
widest_spread_table

Smallest filtered paired spread slabs:


,Slab ID,Pit ID,Layers,Thickness (cm),Min pathway,Max pathway,Max/min ratio,log10(max/min),D11 range (N·mm)
213,51881_slab_0,51881,3,33.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,25.5,1.41,$2.084 \times 10^{9}$
212,51833_slab_0,51833,3,33.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,25.5,1.41,$2.084 \times 10^{9}$
410,989_slab_0,989,2,15.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,26.5,1.42,$1.745 \times 10^{8}$
327,67554_slab_0,67554,1,46.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,27.5,1.44,$4.842 \times 10^{9}$
304,65990_slab_1,65990,1,10.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,27.5,1.44,$4.975 \times 10^{7}$
55,20027_slab_0,20027,1,20.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,27.5,1.44,$3.980 \times 10^{8}$
179,46644_slab_0,46644,1,10.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,27.5,1.44,$4.975 \times 10^{7}$
284,61805_slab_0,61805,1,15.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,27.5,1.44,$1.679 \times 10^{8}$


Widest filtered paired spread slabs:


,Slab ID,Pit ID,Layers,Thickness (cm),Min pathway,Max pathway,Max/min ratio,log10(max/min),D11 range (N·mm)
117,36842_slab_0,36842,1,17.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,120.4,2.08,$6.098 \times 10^{7}$
85,27975_slab_0,27975,1,15.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,120.4,2.08,$4.189 \times 10^{7}$
256,57030_slab_0,57030,1,25.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,120.4,2.08,$1.939 \times 10^{8}$
384,74305_slab_0,74305,1,38.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,120.4,2.08,$6.811 \times 10^{8}$
380,736_slab_0,736,1,13.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,120.4,2.08,$2.727 \times 10^{7}$
376,72548_slab_1,72548,1,40.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,120.4,2.08,$7.944 \times 10^{8}$
375,72548_slab_0,72548,1,40.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,120.4,2.08,$7.944 \times 10^{8}$
277,6027_slab_1,6027,1,9.0,Kim T2 -> Schottner -> Kochle,Geldsetzer -> Wautier -> Kochle,120.4,2.08,$9.048 \times 10^{6}$


## Attribute Correlations With Paired Pathway Spread

Use Spearman rank correlations to identify slab attributes that align most strongly with paired pathway spread, measured as `log10(max D11 / min D11)` within the same slab.

In [26]:
attribute_labels = {
    'n_layers': 'Number of layers',
    'total_thickness_cm': 'Total slab thickness',
    'layer_thickness_mean_cm': 'Mean layer thickness',
    'layer_thickness_std_cm': 'Layer thickness std',
    'layer_thickness_cv': 'Layer thickness CV',
    'layer_thickness_max_cm': 'Maximum layer thickness',
    'hand_hardness_index_mean': 'Mean hand hardness index',
    'hand_hardness_index_std': 'Hand hardness index std',
    'hand_hardness_index_range': 'Hand hardness index range',
    'hand_hardness_index_cv': 'Hand hardness index CV',
    'hand_hardness_index_weighted_mean': 'Thickness-weighted hand hardness',
}

correlation_rows = []
for column, label in attribute_labels.items():
    subset = paired_spread_filtered[[column, 'log10_pathway_spread']].replace([np.inf, -np.inf], np.nan).dropna()
    if len(subset) < 3 or subset[column].nunique() < 2:
        continue
    result = spearmanr(subset[column], subset['log10_pathway_spread'])
    if not math.isfinite(result.statistic):
        continue
    correlation_rows.append(
        {
            'Attribute': label,
            'Spearman rho': float(result.statistic),
            'Abs Spearman rho': abs(float(result.statistic)),
            'p value': float(result.pvalue),
            'n': len(subset),
        }
    )

attribute_correlations = (
    pd.DataFrame(correlation_rows)
    .sort_values(['Abs Spearman rho', 'Attribute'], ascending=[False, True])
    .reset_index(drop=True)
)

attribute_correlation_table = pd.DataFrame(
    {
        'Attribute': attribute_correlations['Attribute'],
        'Spearman rho': attribute_correlations['Spearman rho'].map(lambda value: f'{value:.2f}'),
        'p value': attribute_correlations['p value'].map(lambda value: '<0.001' if value < 0.001 else f'{value:.3f}'),
        'n': attribute_correlations['n'].map(lambda value: f'{int(value):,}'),
    }
)

correlation_paths = save_paper_figure(
    build_d11_spread_attribute_correlations_figure(
        attribute_correlations,
        top_n=10,
        xlabel=r'Spearman $\rho$ with $\log_{10}(D_{11,max}/D_{11,min})$',
    ),
    'd11_paired_spread_attribute_correlations',
    close=True,
)

print('Correlation figure exports:', correlation_paths)
attribute_correlation_table.head(10)

Correlation figure exports: {'repo_pdf': PosixPath('/Users/marykate/Desktop/Snow/SnowPyt-MechParams/paper/figures/d11_paired_spread_attribute_correlations.pdf'), 'repo_png': PosixPath('/Users/marykate/Desktop/Snow/SnowPyt-MechParams/paper/figures/d11_paired_spread_attribute_correlations.png'), 'external_pdf': PosixPath('/Users/marykate/Desktop/Snow/mech_params_paper/figures/d11_paired_spread_attribute_correlations.pdf'), 'external_png': PosixPath('/Users/marykate/Desktop/Snow/mech_params_paper/figures/d11_paired_spread_attribute_correlations.png'), 'pdf': PosixPath('/Users/marykate/Desktop/Snow/SnowPyt-MechParams/paper/figures/d11_paired_spread_attribute_correlations.pdf'), 'png': PosixPath('/Users/marykate/Desktop/Snow/SnowPyt-MechParams/paper/figures/d11_paired_spread_attribute_correlations.png'), 'external_dir': PosixPath('/Users/marykate/Desktop/Snow/mech_params_paper/figures')}


,Attribute,Spearman rho,p value,n
0,Mean hand hardness index,-0.96,<0.001,384
1,Thickness-weighted hand hardness,-0.95,<0.001,384
2,Total slab thickness,0.16,0.001,385
3,Maximum layer thickness,0.14,0.004,385
4,Hand hardness index CV,0.13,0.009,384
5,Layer thickness std,0.13,0.014,385
6,Layer thickness CV,0.12,0.020,385
7,Number of layers,0.11,0.030,385
8,Hand hardness index range,0.11,0.038,384
9,Hand hardness index std,0.11,0.039,384


## Copy-Ready LaTeX Tables

In [27]:
print('Top-8 pathway coverage table:')
print(notebook_latex(coverage_table))
print()
print('Paired pathway effects relative to within-slab median:')
print(notebook_latex(paired_effect_table))
print()
print('Smallest filtered paired spread slabs:')
print(notebook_latex(smallest_spread_table))
print()
print('Widest filtered paired spread slabs:')
print(notebook_latex(widest_spread_table))
print()
print('Attribute correlations with paired pathway spread:')
print(notebook_latex(attribute_correlation_table.head(10)))

Top-8 pathway coverage table:
\begin{tabular}{lll}
\toprule
Pathway & Successful slabs & Coverage (%) \\
\midrule
Kim and Jamieson Table 2 -> Schottner et al. (2026) -> Kochle and Schneebeli (2014) & 743 & 5.0 \\
Kim and Jamieson Table 2 -> Wautier et al. (2015) -> Kochle and Schneebeli (2014) & 743 & 5.0 \\
Geldsetzer and Jamieson (2000) -> Schottner et al. (2026) -> Kochle and Schneebeli (2014) & 740 & 5.0 \\
Geldsetzer and Jamieson (2000) -> Wautier et al. (2015) -> Kochle and Schneebeli (2014) & 740 & 5.0 \\
Geldsetzer and Jamieson (2000) -> Kochle and Schneebeli (2014) -> Kochle and Schneebeli (2014) & 733 & 5.0 \\
Kim and Jamieson Table 2 -> Kochle and Schneebeli (2014) -> Kochle and Schneebeli (2014) & 634 & 4.3 \\
Kim and Jamieson Table 2 -> Bergfeld et al. (2023) -> Kochle and Schneebeli (2014) & 498 & 3.4 \\
Geldsetzer and Jamieson (2000) -> Bergfeld et al. (2023) -> Kochle and Schneebeli (2014) & 475 & 3.2 \\
\bottomrule
\end{tabular}


Paired pathway effects relative to wit